# Explore NEXRAD
### Download and view real weather radar data — any station, any day

*Sebastián Torres · CIWRO, University of Oklahoma*
*Clouds4Africa · July 2026*

---

**What this notebook does.** The five *Reading the Radar* notebooks used a handful of pre-chosen storms as examples. This notebook lets you pick **any U.S. NEXRAD station and any day** back to the early 1990s, download the real radar volume, and plot it yourself — reflectivity, velocity, and (for radars from 2011 onward) the dual-polarization variables from Block 5.

**Before you start — this one is different from Blocks 1–5.** Those notebooks run entirely offline in your browser. This one needs a real internet connection and a full Python environment, because it downloads live data from a public NOAA/Unidata archive and reads it with a specialized radar library. The easiest way to run it is **Google Colab** — free, nothing to install, works from any browser:

1. Go to [colab.research.google.com](https://colab.research.google.com), sign in with any Google account, and choose **File → Upload notebook** to upload this file. (Or open it directly from GitHub if it's hosted there.)
2. Run the cells **from top to bottom** — each one builds on the last.
3. The very first code cell installs some packages. The **first time** you run it, Colab may show a button saying "Restart session" — click it, then just continue running the cells below from where you left off (no need to re-run the install cell).
4. Everything after that is buttons and dropdowns — no coding required.

**How to use it.** Work through the four steps in order: pick a station and day, load a scan, plot a product and tilt, then save the image. There's a bonus step at the end for comparing several tilts at once, just like the four-panel figures in Block 1.

<sub>© 2026 Sebastián Torres · Licensed under CC BY-NC 4.0 · Contact: sebas@ou.edu · Data: NOAA NEXRAD Level II, distributed by NSF Unidata on AWS. NOAA requests attribution when you use or share this data, but does not endorse this notebook.</sub>

In [ ]:
# ── Setup: install and import everything we need ────────────────────────────
# This can take a minute the first time. If Colab prompts you to "Restart
# session" afterward, click it — then just carry on running the cells below.

%pip install -q arm-pyart s3fs fsspec ipywidgets

import gzip
import io
import re
import warnings
from datetime import date, timedelta

import numpy as np
import matplotlib.pyplot as plt
import fsspec
import pyart
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")  # Py-ART is chatty about missing metadata on older files

# Anonymous, read-only connection to the public NEXRAD archive on AWS.
fs = fsspec.filesystem("s3", anon=True)
BUCKET = "unidata-nexrad-level2"

print("Ready. Packages loaded and connected to the NEXRAD archive.")

## Step 1 · Pick a station and a day

Type a 4-letter NEXRAD station code (e.g. `KTLX` for Oklahoma City, `KNKX` for San Diego) and a date, then click **Find scans on this day**. Not sure which code to use? The [NWS radar station map](https://www.roc.noaa.gov/branches/meteorological-services-branch/network-map) shows every site and its 4-letter ID — station codes for the continental U.S. start with `K`.

**Quick start:** the preset dropdown below jumps straight to the same storms used in Blocks 1–5 of the course, so you can try the tool on data you've already seen before exploring on your own.

In [ ]:
# ── Widgets for Step 1 ────────────────────────────────────────────────────────

PRESETS = {
    "— choose a course example (optional) —": None,
    "Block 1 & 4 — KNKX, 11 Mar 2006 (terrain blockage / attenuation)": ("KNKX", date(2006, 3, 11)),
    "Block 2 — KTLX, 20 May 2013 (Moore, OK supercell, near)":          ("KTLX", date(2013, 5, 20)),
    "Block 2 — KINX, 20 May 2013 (same storm, far radar)":              ("KINX", date(2013, 5, 20)),
    "Block 3 — KAKQ, 18 Sep 2003 (Hurricane Isabel)":                   ("KAKQ", date(2003, 9, 18)),
    "Block 5 — KTLX, 20 May 2013 (dual-pol view of the same supercell)": ("KTLX", date(2013, 5, 20)),
}

preset_dd = widgets.Dropdown(options=list(PRESETS.keys()), description="Preset:",
                              layout=widgets.Layout(width="520px"),
                              style={"description_width": "60px"})

station_box = widgets.Text(value="KTLX", description="Station:",
                            placeholder="e.g. KTLX",
                            layout=widgets.Layout(width="220px"),
                            style={"description_width": "60px"})

year_box  = widgets.BoundedIntText(value=2013, min=1991, max=date.today().year,
                                    description="Year:", layout=widgets.Layout(width="160px"))
month_box = widgets.BoundedIntText(value=5, min=1, max=12,
                                    description="Month:", layout=widgets.Layout(width="160px"))
day_box   = widgets.BoundedIntText(value=20, min=1, max=31,
                                    description="Day:", layout=widgets.Layout(width="160px"))

find_button = widgets.Button(description="Find scans on this day", button_style="primary",
                              icon="search", layout=widgets.Layout(width="240px"))
find_output = widgets.Output()

# global state carried between steps
_state = {"folder": None, "files": [], "keys_by_label": {}, "radar": None}

def _on_preset_change(change):
    choice = PRESETS.get(change["new"])
    if choice:
        station_box.value, d = choice[0], choice[1]
        year_box.value, month_box.value, day_box.value = d.year, d.month, d.day

preset_dd.observe(_on_preset_change, names="value")

def _find_scans(_):
    with find_output:
        clear_output()
        station = station_box.value.strip().upper()
        try:
            d = date(year_box.value, month_box.value, day_box.value)
        except ValueError:
            print("That's not a real calendar date — check the year/month/day.")
            return

        folder = f"{BUCKET}/{d.year:04d}/{d.month:02d}/{d.day:02d}/{station}/"
        print(f"Looking in {folder} ...")
        try:
            listing = fs.ls(folder)
        except FileNotFoundError:
            print(
                f"No archive folder for {station} on {d.isoformat()}.\n"
                "Double check the station code (4 letters, e.g. KTLX) and the date — "
                "or the radar may have been offline that day. Try a nearby date."
            )
            return
        except Exception as e:
            print(f"Could not reach the archive right now ({e}). Check your internet connection and try again.")
            return

        pattern = re.compile(r"^(K[A-Z0-9]{3})(\d{8})_(\d{6})(_V0\d)?(\.gz)?$")
        found = []
        for path in listing:
            name = path.split("/")[-1]
            m = pattern.match(name)
            if m:
                hh, mm, ss = m.group(3)[0:2], m.group(3)[2:4], m.group(3)[4:6]
                found.append((f"{hh}:{mm}:{ss} UTC", path))
        found.sort()

        if not found:
            print(f"That folder exists but contains no readable volume scans for {station} on {d.isoformat()}.")
            return

        _state["folder"] = folder
        _state["files"] = found
        _state["keys_by_label"] = {label: path for label, path in found}

        print(f"Found {len(found)} volume scan(s) for {station} on {d.isoformat()}.")
        print("Continue to Step 2 to pick one and load it.")

find_button.on_click(_find_scans)

display(widgets.VBox([
    preset_dd,
    widgets.HBox([station_box, year_box, month_box, day_box]),
    find_button,
    find_output,
]))

## Step 2 · Load one of those scans

Pick a time from the dropdown (it fills in once Step 1 finds some scans) and click **Load this scan**. A volume scan is everything the radar collected in one full rotation through all its tilts — typically 4–10 MB, so this takes a few seconds.

In [ ]:
# ── Widgets for Step 2 ────────────────────────────────────────────────────────

scan_dd = widgets.Dropdown(description="Volume scan:", options=["(run Step 1 first)"],
                            layout=widgets.Layout(width="420px"),
                            style={"description_width": "100px"})
load_button = widgets.Button(description="Load this scan", button_style="primary",
                              icon="download", layout=widgets.Layout(width="200px"))
load_output = widgets.Output()

def _refresh_scan_options():
    if _state["files"]:
        scan_dd.options = [label for label, _ in _state["files"]]
    else:
        scan_dd.options = ["(run Step 1 first)"]

# Re-populate the dropdown right before this cell's widgets are shown, and
# also whenever Step 1's button is clicked again.
_refresh_scan_options()
find_button.on_click(lambda _: _refresh_scan_options())

FRIENDLY_FIELDS = {
    "reflectivity":              "Reflectivity (Z)",
    "velocity":                  "Radial velocity",
    "spectrum_width":            "Spectrum width",
    "differential_reflectivity": "Differential reflectivity (ZDR)",
    "cross_correlation_ratio":   "Correlation coefficient (\u03c1hv)",
    "differential_phase":        "Differential phase (\u03a6DP)",
}

def _load_scan(_):
    with load_output:
        clear_output()
        if not _state["files"]:
            print("Run Step 1 first to find some scans.")
            return
        key = _state["keys_by_label"][scan_dd.value]
        print(f"Downloading and reading {key.split('/')[-1]} ...")
        try:
            with fs.open(key, "rb") as fh:
                raw = fh.read()
            if raw[:2] == b"\x1f\x8b":  # gzip magic bytes — older archives are gzipped
                raw = gzip.decompress(raw)
            radar = pyart.io.read_nexrad_archive(io.BytesIO(raw))
        except Exception as e:
            print(f"Could not read that file ({e}). Try a different scan or station.")
            return

        _state["radar"] = radar
        n_sweeps = radar.nsweeps
        angles = ", ".join(f"{a:.2f}\u00b0" for a in sorted(set(np.round(radar.fixed_angle["data"], 2))))
        available = [FRIENDLY_FIELDS.get(f, f.replace("_", " ").title()) for f in radar.fields]

        print(f"Loaded. {n_sweeps} tilts in this volume, at elevation angles: {angles}")
        print(f"Variables available: {', '.join(available)}")
        print("\nContinue to Step 3 to choose what to plot.")

load_button.on_click(_load_scan)

display(widgets.VBox([scan_dd, load_button, load_output]))

## Step 3 · Choose a product and a tilt, then plot

Pick a variable (only the ones actually present in this scan are listed — dual-pol variables only exist for scans from 2011 onward) and a tilt, then click **Plot**. This uses the same range rings, azimuth spokes, and color scales as the figures in the *Reading the Radar* notebooks.

In [ ]:
# ── Widgets for Step 3 ────────────────────────────────────────────────────────

# label -> (colormap, vmin, vmax, colorbar label)
PRODUCT_STYLE = {
    "reflectivity":              ("NWSRef",  -20,  75, "Reflectivity (dBZ)"),
    "velocity":                  ("NWSVel",  -35,  35, "Radial velocity (m/s)"),
    "spectrum_width":            ("NWS_SPW",   0,  10, "Spectrum width (m/s)"),
    "differential_reflectivity": ("RefDiff",  -2,   6, "Differential reflectivity ZDR (dB)"),
    "cross_correlation_ratio":   ("NWS_CC",  0.7, 1.05, "Correlation coefficient (\u03c1hv)"),
    "differential_phase":        ("Carbone17", 0, 180, "Differential phase \u03a6DP (deg)"),
}

product_dd = widgets.Dropdown(description="Product:", options=["(load a scan first)"],
                               layout=widgets.Layout(width="360px"),
                               style={"description_width": "70px"})
tilt_dd = widgets.Dropdown(description="Tilt:", options=["(pick a product first)"],
                            layout=widgets.Layout(width="240px"),
                            style={"description_width": "50px"})
range_slider = widgets.IntSlider(value=250, min=50, max=460, step=10, description="Max range (km):",
                                  style={"description_width": "120px"},
                                  layout=widgets.Layout(width="420px"))
plot_button = widgets.Button(description="Plot", button_style="success",
                              icon="bar-chart", layout=widgets.Layout(width="140px"))
plot_output = widgets.Output()

_state["current_fig"] = None

def _valid_sweeps_for_field(radar, field):
    '''Sweep indices (and their elevation angle) that actually have real data for this field.'''
    out = []
    for sweep in range(radar.nsweeps):
        try:
            data = radar.get_field(sweep, field)
        except Exception:
            continue
        if np.ma.is_masked(data) and data.mask.all():
            continue
        if not np.ma.is_masked(data) and np.all(np.isnan(data)):
            continue
        out.append((sweep, float(radar.fixed_angle["data"][sweep])))
    return out

def _refresh_product_options():
    radar = _state["radar"]
    if radar is None:
        product_dd.options = ["(load a scan first)"]
        return
    labels = []
    for f in radar.fields:
        labels.append(FRIENDLY_FIELDS.get(f, f.replace("_", " ").title()))
    # keep a lookup from friendly label back to the raw field name
    _state["field_by_label"] = {FRIENDLY_FIELDS.get(f, f.replace("_", " ").title()): f for f in radar.fields}
    product_dd.options = labels

load_button.on_click(lambda _: _refresh_product_options())

def _refresh_tilt_options(change=None):
    radar = _state["radar"]
    if radar is None or product_dd.value not in _state.get("field_by_label", {}):
        tilt_dd.options = ["(pick a product first)"]
        return
    field = _state["field_by_label"][product_dd.value]
    sweeps = _valid_sweeps_for_field(radar, field)
    if not sweeps:
        tilt_dd.options = ["(no data for this product)"]
        return
    _state["sweeps_for_product"] = sweeps
    tilt_dd.options = [f"{angle:.2f}\u00b0  (sweep {sweep})" for sweep, angle in sweeps]

product_dd.observe(_refresh_tilt_options, names="value")

def _plot_ppi(ax, radar, field, sweep, style, max_range):
    '''Draw one PPI panel with range rings and azimuth spokes, Reading-the-Radar style.'''
    cmap, vmin, vmax, cbar_label = style
    disp = pyart.graph.RadarDisplay(radar)
    disp.plot_ppi(field, sweep=sweep, vmin=vmin, vmax=vmax, cmap=cmap,
                  colorbar_flag=False, ax=ax, title_flag=False)
    disp.set_limits(xlim=(-max_range, max_range), ylim=(-max_range, max_range), ax=ax)
    ax.set_aspect("equal")

    rings = list(range(50, int(max_range) + 1, 50))
    if rings:
        disp.plot_range_rings(rings, col="gray", lw=0.5, ls="dotted", ax=ax)
    for az in range(0, 360, 30):
        theta = np.deg2rad(az)
        ax.plot([0, max_range * np.sin(theta)], [0, max_range * np.cos(theta)],
                color="gray", lw=0.5, ls="dashed")

    angle = radar.fixed_angle["data"][sweep]
    ax.set_title(f"{angle:.2f}\u00b0", fontsize=11)
    return disp, cbar_label

def _on_plot(_):
    with plot_output:
        clear_output()
        radar = _state["radar"]
        if radar is None:
            print("Load a scan in Step 2 first.")
            return
        if product_dd.value not in _state.get("field_by_label", {}):
            print("Pick a product first.")
            return
        if not _state.get("sweeps_for_product"):
            print("No usable tilt for this product.")
            return

        idx = tilt_dd.index if isinstance(tilt_dd.index, int) else 0
        sweep, angle = _state["sweeps_for_product"][idx]
        field = _state["field_by_label"][product_dd.value]
        style = PRODUCT_STYLE.get(field, ("viridis", None, None, field))

        fig, ax = plt.subplots(figsize=(7, 7))
        disp, cbar_label = _plot_ppi(ax, radar, field, sweep, style, range_slider.value)
        cbar = plt.colorbar(disp.plots[-1], ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(cbar_label)

        station = radar.metadata.get("instrument_name", "").strip() or "radar"
        try:
            start_time = radar.time["units"].split("since ")[-1].rstrip("Z").replace("T", " ")
        except Exception:
            start_time = ""
        title = f"{station} \u2014 {product_dd.value} \u2014 {angle:.2f}\u00b0 tilt"
        if start_time:
            title += f"\n{start_time} UTC"
        fig.suptitle(title, y=0.98)
        plt.tight_layout()
        _state["current_fig"] = fig
        _state["current_label"] = f"{station}_{field}_{angle:.1f}deg".replace(" ", "_")
        plt.show()

plot_button.on_click(_on_plot)

display(widgets.VBox([
    product_dd,
    widgets.HBox([tilt_dd, plot_button]),
    range_slider,
    plot_output,
]))

## Step 4 · Save your image

Click **Save PNG** to save the plot you just made. In Colab this also downloads it straight to your computer; running locally, it's saved into the same folder as this notebook.

In [ ]:
# ── Widgets for Step 4 ────────────────────────────────────────────────────────

save_button = widgets.Button(description="Save PNG", button_style="info",
                              icon="save", layout=widgets.Layout(width="160px"))
save_output = widgets.Output()

def _on_save(_):
    with save_output:
        clear_output()
        fig = _state.get("current_fig")
        if fig is None:
            print("Make a plot in Step 3 first.")
            return
        filename = f"{_state.get('current_label', 'radar_image')}.png"
        fig.savefig(filename, dpi=150, bbox_inches="tight")
        print(f"Saved as {filename}")
        try:
            from google.colab import files as colab_files
            colab_files.download(filename)
            print("Downloading to your computer...")
        except ImportError:
            print("(Not running in Colab — the file is saved alongside this notebook.)")

save_button.on_click(_on_save)

display(widgets.VBox([save_button, save_output]))

## Bonus · Compare several tilts at once

This reproduces the four-panel, multi-tilt figures from Block 1 — the same storm, same product, at four different elevation angles, so you can watch how blockage, beam height, or a hurricane's velocity field change as the beam climbs. It uses whichever product and station are currently loaded above; only the tilt dropdown is ignored.

In [ ]:
# ── Bonus: 2x2 multi-tilt comparison ─────────────────────────────────────────

compare_button = widgets.Button(description="Compare 4 tilts", button_style="success",
                                 icon="table", layout=widgets.Layout(width="200px"))
compare_output = widgets.Output()

def _on_compare(_):
    with compare_output:
        clear_output()
        radar = _state["radar"]
        if radar is None:
            print("Load a scan in Step 2 first.")
            return
        if product_dd.value not in _state.get("field_by_label", {}):
            print("Pick a product in Step 3 first.")
            return

        field = _state["field_by_label"][product_dd.value]
        style = PRODUCT_STYLE.get(field, ("viridis", None, None, field))
        sweeps = _state.get("sweeps_for_product") or _valid_sweeps_for_field(radar, field)
        if len(sweeps) < 2:
            print("This product only has one usable tilt in this scan — nothing to compare.")
            return

        # pick up to 4 sweeps spread evenly across the available elevation angles
        n_panels = min(4, len(sweeps))
        pick_idx = np.linspace(0, len(sweeps) - 1, n_panels).round().astype(int)
        chosen = [sweeps[i] for i in sorted(set(pick_idx))]
        while len(chosen) < 4:
            chosen.append(chosen[-1])  # pad by repeating the last tilt if fewer than 4 exist

        fig, axes = plt.subplots(2, 2, figsize=(10, 10))
        axes = axes.ravel()
        last_disp = None
        for ax, (sweep, angle) in zip(axes, chosen[:4]):
            last_disp, cbar_label = _plot_ppi(ax, radar, field, sweep, style, range_slider.value)

        plt.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.1, wspace=0.25, hspace=0.25)
        cax = fig.add_axes([0.25, 0.03, 0.5, 0.015])
        cbar = plt.colorbar(last_disp.plots[-1], cax=cax, orientation="horizontal")
        cbar.set_label(cbar_label)

        station = radar.metadata.get("instrument_name", "").strip() or "radar"
        try:
            start_time = radar.time["units"].split("since ")[-1].rstrip("Z").replace("T", " ")
        except Exception:
            start_time = ""
        title = f"{station} \u2014 {product_dd.value} \u2014 tilts compared"
        if start_time:
            title += f"\n{start_time} UTC"
        fig.suptitle(title, y=0.97)

        _state["current_fig"] = fig
        _state["current_label"] = f"{station}_{field}_4tilt_compare".replace(" ", "_")
        plt.show()
        print("Click Save PNG above to save this figure too.")

compare_button.on_click(_on_compare)

display(widgets.VBox([compare_button, compare_output]))

## Tips and troubleshooting

- **"No archive folder for that station/date"** — double-check the 4-letter station code and that the radar was operating that day (a few sites have gaps). The [NWS radar station map](https://www.roc.noaa.gov/branches/meteorological-services-branch/network-map) lists every station by code and location.
- **Dual-pol variables missing** — differential reflectivity, correlation coefficient, and differential phase only exist for scans from **2011 onward**, after the network's dual-polarization upgrade. Older scans (like the 2003 and 2006 examples) will only offer reflectivity, velocity, and spectrum width.
- **A product looks solid one color / empty** — some tilts don't collect every product (common in "split cut" scan strategies, where the lowest couple of tilts alternate between reflectivity-only and velocity-only sweeps). The tilt dropdown in Step 3 only lists tilts that actually have real data for the product you picked, so this shouldn't happen — but if it does, try a different tilt.
- **Slow or failing downloads** — the archive is a public, best-effort service; if a download hangs, just try again or pick a different scan.
- **Restarting midway** — if you re-run Step 1 with a new station/date, just work back down through Steps 2–4 again; nothing carries over automatically except what's in the dropdowns.

**Where this data comes from.** NEXRAD Level II data are collected by NOAA's National Weather Service, FAA, and U.S. Air Force WSR-88D radars, and distributed openly on Amazon S3 by NSF Unidata as part of NOAA's Open Data Dissemination program. The data are free to use; NOAA asks only that you attribute the source and not imply NOAA's endorsement of anything you make with it.